# Phase 2 — IndicTrans2 Fine-tuning (Full Samanantar en→hi)

IndicTrans2 is a pre-trained multilingual translation model from AI4Bharat.
This notebook:
1. Installs and loads IndicTrans2
2. Preprocesses the full Samanantar dataset in the format IndicTrans2 expects
   (different from the RNN/LSTM preprocessing — uses IndicTrans2's own tokenizer)
3. Fine-tunes the model on the full dataset
4. Evaluates with BLEU, ChrF, BERTScore, COMET

### Why separate preprocessing?
IndicTrans2 uses Sentence-Piece BPE tokenisation internally.  You must NOT
pass it your hand-built word-level vocab from the NN preprocessing.  This
notebook runs its own clean → IndicTrans2-tokenise → DataCollator pipeline.

In [ ]:
# ── Cell 1: Install IndicTrans2 dependencies ──────────────────────────────────
# Run this cell once. After installation, restart the kernel and skip this cell.
import subprocess, sys

packages = [
    "git+https://github.com/AI4Bharat/IndicTrans2.git",  # IndicTrans2 repo
    "sacrebleu",
    "bert-score",
    "unbabel-comet",
    "transformers>=4.33",
    "sentencepiece",
    "datasets",
    "accelerate",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("Installation complete. Restart kernel if this is your first run.")

In [ ]:
# ── Cell 2: Imports ───────────────────────────────────────────────────────────
import os
import re
import gc
import math
import glob
import pickle
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from sacrebleu.metrics import BLEU, CHRF
from bert_score import score as bert_score_fn
import evaluate as hf_evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
# ── Cell 3: Config ────────────────────────────────────────────────────────────
# IndicTrans2 model identifier on HuggingFace Hub
# The 200M parameter model is recommended for fine-tuning on a single GPU.
# If you have a GPU with 40+ GB VRAM you can try the 600M model:
#   'ai4bharat/indictrans2-en-indic-1B'
MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'

# IndicTrans2 language codes
SRC_LANG = 'eng_Latn'   # English
TGT_LANG = 'hin_Deva'   # Hindi

# Paths
# NOTE: IndicTrans2 does NOT use the phase2_data/ from the NN preprocessing.
# It has its own tokenizer so we point to the raw CSVs.
RAW_DATA_DIR = './phase2_data/'    # where Samanantar raw chunks are
OUTPUT_DIR   = './phase2_indictrans2_output/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training hyperparameters
# CHANGE: batch sizes are much smaller for a large transformer on one GPU.
# gradient_accumulation_steps effectively gives batch_size=32 while only
# loading 4 samples at a time — safe for 16 GB VRAM GPUs.
PER_DEVICE_TRAIN_BATCH = 4
PER_DEVICE_EVAL_BATCH  = 8
GRAD_ACCUM_STEPS       = 8       # effective batch = 4*8 = 32
N_EPOCHS               = 3       # fine-tuning needs fewer epochs than training from scratch
LEARNING_RATE          = 5e-5    # small LR for fine-tuning pre-trained weights
MAX_SRC_LEN            = 128     # IndicTrans2 supports up to 256 tokens
MAX_TGT_LEN            = 128
WARMUP_STEPS           = 500
SAVE_STEPS             = 2000
EVAL_STEPS             = 2000

print("Config set.")

In [ ]:
# ── Cell 4: Load IndicTrans2 tokenizer and model ──────────────────────────────
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG
)

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
# ── Cell 5: Cleaning (same basic rules as NN preprocessing) ──────────────────
# IndicTrans2 handles its own subword tokenisation, so we only need
# light surface cleaning — NOT the aggressive word-level cleaning from
# the NN preprocessing (which stripped everything except letters+punctuation).
# We keep numbers, special punctuation, etc. so the model sees realistic text.

def clean_english_it2(text):
    """Light cleaning for IndicTrans2: lowercase, normalise whitespace."""
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def clean_hindi_it2(text):
    """Light cleaning for IndicTrans2: keep Devanagari + common punctuation."""
    text = str(text).strip()
    # Remove characters that are neither Devanagari, ASCII printable, nor
    # common punctuation. This removes stray encoding artefacts.
    text = re.sub(r'[^\u0900-\u097F\u0020-\u007E।॥]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Sanity check
print(clean_english_it2("  Hello, World! 123  "))
print(clean_hindi_it2("नमस्ते दुनिया! Hello 123"))

In [ ]:
# ── Cell 6: Load and tokenise data from raw chunks ────────────────────────────
# IndicTrans2 uses HuggingFace Datasets format.
# We load all raw chunks for each split, clean them, and tokenise in batches.
# The tokenised dataset is saved to disk so it only needs to be done once.

def load_raw_split(split_name):
    """Read all raw CSV chunks for a split into one DataFrame."""
    chunk_files = sorted(glob.glob(
        os.path.join(RAW_DATA_DIR, f'{split_name}_raw_chunk*.csv')
    ))
    assert chunk_files, f"No raw chunks found for '{split_name}' in {RAW_DATA_DIR}"
    dfs = []
    for f in chunk_files:
        df = pd.read_csv(f)
        df['english'] = df['english'].apply(clean_english_it2)
        df['hindi']   = df['hindi'].apply(clean_hindi_it2)
        # Drop empty rows after cleaning
        df = df[(df['english'].str.len() > 0) & (df['hindi'].str.len() > 0)]
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    print(f"  {split_name}: {len(combined):,} rows after cleaning")
    return combined

print("Loading raw data splits...")
train_df = load_raw_split('train')
val_df   = load_raw_split('val')
test_df  = load_raw_split('test')

In [ ]:
# ── Cell 7: Tokenisation for IndicTrans2 ──────────────────────────────────────
# IndicTrans2 expects the tokenizer to be called with:
#   text = source sentences (English)
#   text_target = target sentences (Hindi)
# It adds its own language tags internally.

def tokenise_function(batch):
    """Tokenise a batch of (english, hindi) pairs for IndicTrans2."""
    model_inputs = tokenizer(
        batch['english'],
        text_target=batch['hindi'],
        max_length=MAX_SRC_LEN,
        truncation=True,
        padding=False    # padding is done per-batch by DataCollatorForSeq2Seq
    )
    return model_inputs

def df_to_hf_dataset(df):
    """Convert a pandas DataFrame to a HuggingFace Dataset and tokenise it."""
    hf = HFDataset.from_pandas(
        df[['english', 'hindi']].reset_index(drop=True)
    )
    tokenised = hf.map(
        tokenise_function,
        batched=True,
        batch_size=1000,
        remove_columns=['english', 'hindi'],
        desc="Tokenising"
    )
    return tokenised

print("Tokenising train split... (this takes a while for 8M+ pairs)")
train_tokenised = df_to_hf_dataset(train_df)
print("Tokenising val split...")
val_tokenised   = df_to_hf_dataset(val_df)
print("Tokenising test split...")
test_tokenised  = df_to_hf_dataset(test_df)

# Free DataFrames — they're large
del train_df, val_df
gc.collect()

print(f"\nTrain tokenised: {len(train_tokenised):,} examples")
print(f"Val tokenised:   {len(val_tokenised):,} examples")
print(f"Test tokenised:  {len(test_tokenised):,} examples")

In [ ]:
# ── Cell 8: Save tokenised datasets to disk ───────────────────────────────────
# Saving means we can reload quickly if the notebook restarts.
TOKENISED_DIR = os.path.join(OUTPUT_DIR, 'tokenised_datasets')
train_tokenised.save_to_disk(os.path.join(TOKENISED_DIR, 'train'))
val_tokenised.save_to_disk(os.path.join(TOKENISED_DIR, 'val'))
test_tokenised.save_to_disk(os.path.join(TOKENISED_DIR, 'test'))
print("Tokenised datasets saved.")

In [ ]:
# ── Cell 8b: (Re-)load tokenised datasets — run this cell after a restart ─────
# If you restart the kernel after Cell 8, run this cell instead of re-tokenising.
# Comment out Cell 7 and Cell 8 to skip re-tokenisation.
from datasets import load_from_disk
TOKENISED_DIR = os.path.join(OUTPUT_DIR, 'tokenised_datasets')
train_tokenised = load_from_disk(os.path.join(TOKENISED_DIR, 'train'))
val_tokenised   = load_from_disk(os.path.join(TOKENISED_DIR, 'val'))
test_tokenised  = load_from_disk(os.path.join(TOKENISED_DIR, 'test'))
print(f"Loaded — Train: {len(train_tokenised):,} | Val: {len(val_tokenised):,} | Test: {len(test_tokenised):,}")

In [ ]:
# ── Cell 9: Data collator ─────────────────────────────────────────────────────
# DataCollatorForSeq2Seq pads each batch to the longest sequence in that
# batch (not to a global MAX_LEN). This is much more efficient than
# pre-padding everything to MAX_LEN as the NN preprocessing did.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,   # -100 tells CrossEntropyLoss to ignore padding
    pad_to_multiple_of=8       # pads to multiples of 8 for tensor core efficiency
)
print("Data collator ready.")

In [ ]:
# ── Cell 10: BLEU metric for training-time eval ───────────────────────────────
# HuggingFace Trainer calls compute_metrics after every eval_steps.
# We compute sacrebleu BLEU here so we can see quality progress during training,
# not just loss.

sacrebleu_metric = hf_evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Clip out-of-range token ids (can appear on first few steps)
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)

    # Replace -100 with pad_token_id in labels before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # sacrebleu expects list-of-strings for preds and list-of-list-of-strings for refs
    result = sacrebleu_metric.compute(
        predictions=decoded_preds,
        references=[[lbl] for lbl in decoded_labels]
    )
    return {'bleu': round(result['score'], 2)}

print("compute_metrics defined.")

In [ ]:
# ── Cell 11: TrainingArguments ────────────────────────────────────────────────
# CHANGE: all Phase 1 NN models used a manual training loop.
# For IndicTrans2 we use HuggingFace Trainer which handles:
#   - gradient accumulation (so effective batch = PER_DEVICE * GRAD_ACCUM)
#   - mixed precision (fp16) for 2× speedup on modern GPUs
#   - automatic checkpoint saving and resuming
#   - evaluation at regular intervals

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training schedule
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,

    # Optimiser
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,

    # Speed
    fp16=torch.cuda.is_available(),  # mixed precision — set False if you get NaN losses
    dataloader_num_workers=4,

    # Evaluation & saving
    evaluation_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,              # keep only the 3 most recent checkpoints to save disk
    load_best_model_at_end=True,
    metric_for_best_model='bleu',
    greater_is_better=True,

    # Generation settings for eval
    predict_with_generate=True,
    generation_max_length=MAX_TGT_LEN,

    # Logging
    logging_dir=os.path.join(OUTPUT_DIR, 'logs'),
    logging_steps=200,
    report_to='none',   # change to 'wandb' if you have W&B set up
)
print("TrainingArguments set.")

In [ ]:
# ── Cell 12: Trainer ──────────────────────────────────────────────────────────
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenised,
    eval_dataset=val_tokenised,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        # EarlyStopping: stop if BLEU doesn't improve for 3 eval rounds
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)
print("Trainer ready.")

In [ ]:
# ── Cell 13: Fine-tune ────────────────────────────────────────────────────────
# CHANGE: HuggingFace Trainer auto-resumes from the latest checkpoint in
# OUTPUT_DIR if one exists — no extra code needed for resume.
# If OUTPUT_DIR has a checkpoint, training resumes from there automatically.
print("Starting fine-tuning...")
print(f"Checkpoints will be saved to: {OUTPUT_DIR}")

trainer.train(
    # Automatically resume from checkpoint if one exists
    resume_from_checkpoint=any(
        f.startswith('checkpoint') for f in os.listdir(OUTPUT_DIR)
    ) if os.path.exists(OUTPUT_DIR) else False
)

print("\nFine-tuning complete!")

# Save final model
FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, 'final_model')
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Final model saved to: {FINAL_MODEL_DIR}")

In [ ]:
# ── Cell 14: Generate translations on test set ────────────────────────────────
print("Generating translations on test set...")

# Use Trainer.predict for batched generation — much faster than looping
predictions_output = trainer.predict(
    test_tokenised,
    metric_key_prefix='test',
    max_length=MAX_TGT_LEN,
    num_beams=4,            # beam search — better quality than greedy
)

pred_ids  = predictions_output.predictions
label_ids = predictions_output.label_ids

# Replace -100 in labels
pred_ids  = np.where(pred_ids  != -100, pred_ids,  tokenizer.pad_token_id)
label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

all_hypotheses = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
all_references = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

print(f"Generated {len(all_hypotheses):,} translations.")

In [ ]:
# ── Cell 15: Compute all metrics ──────────────────────────────────────────────
bleu_metric = BLEU()
chrf_metric = CHRF()

bleu_score = bleu_metric.corpus_score(all_hypotheses, [all_references])
chrf_score = chrf_metric.corpus_score(all_hypotheses, [all_references])

print("Computing BERTScore (using multilingual BERT)...")
_, _, F1_bert = bert_score_fn(
    all_hypotheses, all_references,
    lang='hi',
    model_type='bert-base-multilingual-cased',
    verbose=False
)
bert_f1 = F1_bert.mean().item()

print("\n" + "=" * 60)
print("    IndicTrans2 (FINE-TUNED) — PHASE 2 TEST RESULTS")
print("=" * 60)
print(f"  BLEU      : {bleu_score.score:.2f}")
print(f"  ChrF      : {chrf_score.score:.2f}")
print(f"  BERTScore : {bert_f1:.4f}")
print("=" * 60)

In [ ]:
# ── Cell 16: COMET (optional) ─────────────────────────────────────────────────
try:
    from comet import download_model, load_from_checkpoint

    # Get source sentences for COMET (it needs src, mt, ref)
    test_src_ids = [
        ex['input_ids'] for ex in test_tokenised
    ]
    test_sources = tokenizer.batch_decode(test_src_ids, skip_special_tokens=True)

    comet_model_path = download_model("Unbabel/wmt22-comet-da")
    comet_model      = load_from_checkpoint(comet_model_path)
    comet_data = [
        {'src': s, 'mt': h, 'ref': r}
        for s, h, r in zip(test_sources, all_hypotheses, all_references)
    ]
    comet_score = comet_model.predict(comet_data, batch_size=64, gpus=1)['system_score']
    print(f"COMET: {comet_score:.4f}")
except Exception as e:
    print(f"COMET skipped: {e}")

In [ ]:
# ── Cell 17: Sample translations ──────────────────────────────────────────────
print("SAMPLE TRANSLATIONS — IndicTrans2 Fine-tuned")
print("=" * 65)
for i in range(10):
    print(f"Hypothesis : {all_hypotheses[i]}")
    print(f"Reference  : {all_references[i]}")
    print("-" * 65)

In [ ]:
# ── Cell 18: Interactive translation (test your own sentences) ────────────────
# Load the fine-tuned model from disk (useful after restarting)
from transformers import pipeline

FINAL_MODEL_DIR = os.path.join(OUTPUT_DIR, 'final_model')
translator = pipeline(
    'translation',
    model=FINAL_MODEL_DIR,
    tokenizer=FINAL_MODEL_DIR,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG,
    device=0 if device.type == 'cuda' else -1
)

test_sentences = [
    "The government announced a new policy for education.",
    "She was reading a book in the library.",
    "The train will arrive at the station at six o'clock.",
]

print("Custom translation test:")
print("=" * 65)
for sent in test_sentences:
    result = translator(sent, max_length=MAX_TGT_LEN)
    print(f"English : {sent}")
    print(f"Hindi   : {result[0]['translation_text']}")
    print("-" * 65)